In [1]:
import polars as pl
from utils import getConnection
from utils import cleanContracts
from utils import readCSV

pl.Config.set_tbl_rows(100) # Show up to 100 rows

con, dataset_path = getConnection() # Create the duckDB connection
readCSV(con, dataset_path) # Read CSV and create temp contracts table
cleanContracts(con) # Clean the data before we begin our financial analysis

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Create a condensed temp table, one row represents a contracts latest state

In [2]:
# This simplifies later queries 
con.sql("""CREATE TEMP TABLE contracts_latest AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY reference_number
               ORDER BY contract_date DESC
           ) AS rn
    FROM contracts_clean
) t
WHERE rn = 1;
""")

# Top 10 vendors by contract value

In [3]:
with pl.Config(float_precision=2, thousands_separator=','):
    print(con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY total_earned DESC) AS rank,
    vendor_name,
    total_contracts,
    total_earned
FROM (
SELECT
    vendor_name,
    COUNT(*) AS total_contracts,
    SUM(contract_value) AS total_earned
    FROM contracts_latest
    GROUP BY vendor_name
    ORDER BY total_earned DESC
    ) t
""").pl().limit(10))

shape: (10, 4)
┌──────┬─────────────────────────────────┬─────────────────┬───────────────────┐
│ rank ┆ vendor_name                     ┆ total_contracts ┆ total_earned      │
│ ---  ┆ ---                             ┆ ---             ┆ ---               │
│ i64  ┆ str                             ┆ i64             ┆ f64               │
╞══════╪═════════════════════════════════╪═════════════════╪═══════════════════╡
│ 1    ┆ Irving Shipbuilding Inc         ┆ 4               ┆ 20,402,436,551.91 │
│ 2    ┆ BANCTEC (CANADA), INC.          ┆ 8               ┆ 20,031,143,258.03 │
│ 3    ┆ General Dynamics Land Systems … ┆ 54              ┆ 8,625,542,848.69  │
│ 4    ┆ PCL CONSTRUCTORS CANADA INC.    ┆ 40              ┆ 8,314,979,097.55  │
│ 5    ┆ I.M.P Group Limited             ┆ 3               ┆ 8,166,089,765.70  │
│ 6    ┆ Irving Shipbuilding Inc.        ┆ 1               ┆ 8,010,267,025.00  │
│ 7    ┆ Sikorsky International Operati… ┆ 7               ┆ 7,607,385,308.00  │
│ 8    ┆ Casc

# Top 10 vendors by total contracts

In [4]:
con.sql(""" 
    SELECT
    ROW_NUMBER() OVER (ORDER BY total_contracts DESC) AS rank,
    vendor_name,
    total_contracts,
    ROUND(total_earned, 2) AS total_earned
FROM (
SELECT
    vendor_name,
    COUNT(*) AS total_contracts,
    SUM(contract_value) AS total_earned
    FROM contracts_latest
    GROUP BY vendor_name
    ORDER BY total_contracts DESC
    ) t
""").limit(10)

┌───────┬─────────────────────────────────────────┬─────────────────┬───────────────┐
│ rank  │               vendor_name               │ total_contracts │ total_earned  │
│ int64 │                 varchar                 │      int64      │    double     │
├───────┼─────────────────────────────────────────┼─────────────────┼───────────────┤
│     1 │ MCKESSON CANADA CORPORATION             │            2641 │  142243943.96 │
│     2 │ Veritaaq Technology House Inc.          │            1863 │  743968622.58 │
│     3 │ CANADIAN CORPS OF COMMISSIONAIRES       │            1768 │  413409549.65 │
│     4 │ IMPERIAL OIL                            │            1549 │   91901439.35 │
│     5 │ SIMEX DEFENCE INC. / DEFENSE SIMEX INC. │            1423 │  169525657.12 │
│     6 │ MCKESSON CANADA                         │            1309 │   70572889.64 │
│     7 │ Unisource Technology Inc                │            1052 │    90123529.9 │
│     8 │ SHELL                                   │   

# How many contracts have been amended? 

In [5]:
con.sql(""" 
SELECT
    COUNT(*) AS contracts_with_amendments
FROM (
    SELECT
        reference_number,
        COUNT(*) AS version_count
    FROM
        contracts_clean
    GROUP BY
        reference_number
    HAVING
        COUNT(*) > 1   -- if the count is more than 1, it means there are amendments
) AS amended_contracts;
""")

┌───────────────────────────┐
│ contracts_with_amendments │
│           int64           │
├───────────────────────────┤
│                    143148 │
└───────────────────────────┘

# Top 10 contracts by amendment count

In [6]:
con.sql("""
    SELECT
    reference_number,
    MAX(vendor_name) AS vendor_name, -- same for all versions
    ROUND(SUM(COALESCE(amendment_value, 0)), 2) AS total_amended,
    COUNT(*) - 1 AS num_amendments, -- # of amendments (exclude original)
    ROUND(AVG(COALESCE(amendment_value, 0)), 2) AS avg_amendment_value
FROM
    contracts_clean -- BACK TO FULL TABLE TO GET ALL VERSIONS
GROUP BY
    reference_number
HAVING
    COUNT(*) > 1 -- only amended contracts
ORDER BY
    num_amendments DESC;
""").limit(10)

┌──────────────────────┬────────────────────────────────────┬───────────────┬────────────────┬─────────────────────┐
│   reference_number   │            vendor_name             │ total_amended │ num_amendments │ avg_amendment_value │
│       varchar        │              varchar               │    double     │     int64      │       double        │
├──────────────────────┼────────────────────────────────────┼───────────────┼────────────────┼─────────────────────┤
│ C-2025-2026-Q2-00001 │ XEROX CANADA INC.                  │  792341763.51 │             74 │         10564556.85 │
│ C-2023-2024-Q2-00001 │ Xerox Canada Ltd.                  │    3644995.32 │             74 │            48599.94 │
│ C-2022-2023-Q2-00001 │ XEROX CANADA INC.                  │   12180819.23 │             74 │           162410.92 │
│ C-2024-2025-Q1-00001 │ ZABELLE INC.                       │  105562822.68 │             74 │           1407504.3 │
│ C-2024-2025-Q3-00001 │ WSP Canada Inc.                    │  7

# Top 10 contracts by amendment impact

In [7]:
con.sql("""
    -- Amendment impact using contracts_latest (one row per contract)
    SELECT
        reference_number,
        vendor_name,
        ROUND(contract_value - original_value, 2) AS amendment_impact
    FROM
        contracts_latest
    ORDER BY
        amendment_impact DESC
    LIMIT 10;
""")

┌───────────────────────┬──────────────────────────────────────┬──────────────────┐
│   reference_number    │             vendor_name              │ amendment_impact │
│        varchar        │               varchar                │      double      │
├───────────────────────┼──────────────────────────────────────┼──────────────────┤
│ C-2024-2025-Q2-02238  │ BANCTEC (CANADA), INC.               │   20012319127.56 │
│ C-2025-2026-Q1-04643  │ Irving Shipbuilding Inc              │     3516001276.9 │
│ C-2023-2024-Q4-005303 │ I.M.P Group Limited                  │    3292160898.11 │
│ C-2023-2024-Q4-005304 │ Irving Shipbuilding Inc              │     3200725032.5 │
│ C-2023-2024-Q1-04775  │ Irving Shipbuilding Inc              │    2577923398.34 │
│ C-2024-2025-Q1-04719  │ L-3 Communications MAS (Canada) Inc. │     2296174597.0 │
│ C-2022-2023-Q4-04984  │ I.M.P Group Limited                  │    2254550863.55 │
│ C-2022-2023-Q4-04981  │ L-3 Communications MAS (Canada) Inc. │     2195013

# Top 10 goods and services

In [8]:
con.sql("""
    -- Count of contracts by commodity code
    SELECT
        commodity_code,
        COUNT(*) AS num_contracts,
        ROUND(SUM(contract_value), 2) AS total_value
    FROM
        contracts_latest
    WHERE
        commodity_code IS NOT NULL
    GROUP BY
        commodity_code
    ORDER BY
        num_contracts DESC
""").limit(10)

┌────────────────┬───────────────┬───────────────┐
│ commodity_code │ num_contracts │  total_value  │
│    varchar     │     int64     │    double     │
├────────────────┼───────────────┼───────────────┤
│ N9130E         │          8293 │  526708301.24 │
│ D302A          │          4285 │ 4634028256.71 │
│ N8900          │          4074 │   69921648.91 │
│ R019           │          3960 │ 1016803392.02 │
│ N6505          │          3777 │  369573795.48 │
│ D302           │          3391 │ 5853373068.88 │
│ 0              │          3171 │  186442373.61 │
│ M190A          │          3148 │  450891095.25 │
│ N7110          │          2896 │  103811500.25 │
│ 78182000       │          2820 │  916014894.76 │
├────────────────┴───────────────┴───────────────┤
│ 10 rows                              3 columns │
└────────────────────────────────────────────────┘

# Top 10 postal codes, where is most of the work being done?

In [13]:
con.sql("""
    SELECT    
        ROW_NUMBER() OVER (ORDER BY postal_code_count DESC) AS rank,
        vendor_postal_code,
        postal_code_count
    FROM ( 
        SELECT vendor_postal_code, COUNT(*) AS postal_code_count
        FROM contracts_latest
        GROUP BY vendor_postal_code
        ORDER BY postal_code_count DESC
    ) t
""").limit(11) # Include one extra row to account for NULL

┌───────┬────────────────────┬───────────────────┐
│ rank  │ vendor_postal_code │ postal_code_count │
│ int64 │      varchar       │       int64       │
├───────┼────────────────────┼───────────────────┤
│     1 │ NULL               │            396821 │
│     2 │ M5W                │              5225 │
│     3 │ K1P                │              3731 │
│     4 │ B3B                │              2375 │
│     5 │ K1G                │              2033 │
│     6 │ T2P                │              2007 │
│     7 │ K2E                │              1424 │
│     8 │ K1Z                │              1362 │
│     9 │ H3C                │              1265 │
│    10 │ K2P                │              1248 │
│    11 │ K1N                │               970 │
├───────┴────────────────────┴───────────────────┤
│ 11 rows                              3 columns │
└────────────────────────────────────────────────┘